# Full DeepGlobe pretraining
Trains DeepLabV3+ (ResNet-34) on all 6,226 DeepGlobe images and saves the model to Google Drive after every epoch.
If Colab disconnects, run all cells again: training resumes from the last saved epoch.
Runtime > Change runtime type > T4 GPU. About 1.5 to 2 hours.

In [ ]:
# 1. Setup: Drive, packages, DeepGlobe download (needs kaggle.json the first time)
from google.colab import drive, files
drive.mount("/content/drive")
SAVE = "/content/drive/MyDrive/road_models"
import os; os.makedirs(SAVE, exist_ok=True)
!pip -q install segmentation-models-pytorch kaggle
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    if os.path.exists(f"{SAVE}/kaggle.json"):
        !mkdir -p ~/.kaggle && cp "{SAVE}/kaggle.json" ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    else:
        files.upload()   # pick kaggle.json once; a copy is kept in Drive for next time
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json && cp ~/.kaggle/kaggle.json "{SAVE}/"
if not os.path.isdir("/content/dg/train"):
    !kaggle datasets download -d balraj98/deepglobe-road-extraction-dataset -p /content/dg -q
    !cd /content/dg && unzip -q -o '*.zip' 'train/*'
!ls /content/dg/train | wc -l

In [ ]:
# 2. Data: all images, 300 kept aside for validation, random 512 px crops (keeps 0.5 m resolution)
import glob, random, numpy as np, torch
from PIL import Image
random.seed(0); np.random.seed(0); torch.manual_seed(0)
sats = sorted(glob.glob("/content/dg/train/*_sat.jpg"))
random.shuffle(sats)
val_files, train_files = sats[:300], sats[300:]
print(len(train_files), "training images |", len(val_files), "validation images")
MEAN = np.array([0.485, 0.456, 0.406]); STD = np.array([0.229, 0.224, 0.225])
class DG(torch.utils.data.Dataset):
    def __init__(self, fs, train): self.fs, self.train = fs, train
    def __len__(self): return len(self.fs)
    def __getitem__(self, i):
        s = self.fs[i]
        img = np.array(Image.open(s).convert("RGB"))
        m = (np.array(Image.open(s.replace("_sat.jpg", "_mask.png")).convert("L")) > 128).astype(np.float32)
        if self.train:
            y0, x0 = np.random.randint(0, img.shape[0] - 512, 2)
            img, m = img[y0:y0+512, x0:x0+512], m[y0:y0+512, x0:x0+512]
            k = np.random.randint(4); img, m = np.rot90(img, k), np.rot90(m, k)
            if np.random.rand() < .5: img, m = img[:, ::-1], m[:, ::-1]
        else:
            img, m = img[256:768, 256:768], m[256:768, 256:768]   # fixed center crop for validation
        x = ((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)
        return torch.tensor(x.copy(), dtype=torch.float32), torch.tensor(m.copy())[None]
tr = torch.utils.data.DataLoader(DG(train_files, True), batch_size=8, shuffle=True, num_workers=2, drop_last=True)
va = torch.utils.data.DataLoader(DG(val_files, False), batch_size=8, num_workers=2)

In [ ]:
# 3. Train with checkpoints in Drive (resumes automatically)
import segmentation_models_pytorch as smp, json, time
dev = "cuda"; EPOCHS = 20
model = smp.DeepLabV3Plus("resnet34", encoder_weights="imagenet", classes=1).to(dev)
opt = torch.optim.AdamW(model.parameters(), 1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
scaler = torch.cuda.amp.GradScaler()
dice = smp.losses.DiceLoss("binary"); bce = torch.nn.BCEWithLogitsLoss()
ckpt, log_path, start, log = f"{SAVE}/dg_full_last.pt", f"{SAVE}/dg_full_log.json", 0, []
if os.path.exists(ckpt):
    c = torch.load(ckpt); model.load_state_dict(c["model"]); opt.load_state_dict(c["opt"])
    sched.load_state_dict(c["sched"]); start = c["epoch"] + 1; log = json.load(open(log_path))
    print("Resuming from epoch", start + 1)

def evaluate():
    model.eval(); tp = fp = fn = 0
    with torch.no_grad():
        for x, y in va:
            p = (torch.sigmoid(model(x.to(dev))) > 0.5).float().cpu()
            tp += (p*y).sum().item(); fp += (p*(1-y)).sum().item(); fn += ((1-p)*y).sum().item()
    P, R = tp/(tp+fp+1e-9), tp/(tp+fn+1e-9)
    return dict(precision=P, recall=R, f1=2*P*R/(P+R+1e-9), iou=tp/(tp+fp+fn+1e-9))

best = max([h["f1"] for h in log], default=0)
for ep in range(start, EPOCHS):
    model.train(); tot, t0 = 0, time.time()
    for x, y in tr:
        x, y = x.to(dev), y.to(dev)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(x); loss = dice(out, y) + bce(out, y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item()
    sched.step()
    h = evaluate(); h.update(epoch=ep+1, loss=tot/len(tr), minutes=round((time.time()-t0)/60, 1)); log.append(h)
    print(f"ep {ep+1:2d} | loss {h['loss']:.3f} | val P {h['precision']:.3f} R {h['recall']:.3f} F1 {h['f1']:.3f} IoU {h['iou']:.3f} | {h['minutes']} min")
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": ep}, ckpt)
    json.dump(log, open(log_path, "w"), indent=1)
    if h["f1"] > best:
        best = h["f1"]; torch.save(model.state_dict(), f"{SAVE}/dg_full_best.pt"); print("   saved best model")
print("Done. Best validation F1:", round(best, 3), "| model:", f"{SAVE}/dg_full_best.pt")